In [1]:
# ==============================
# 0. MOUNT GOOGLE DRIVE
# ==============================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ==============================
# 1. INSTALL REQUIRED PACKAGES
# ==============================
!pip install awscli h5py scipy -q

# ==============================
# 2. IMPORTS
# ==============================
import os
import subprocess
from pathlib import Path
import shutil
import h5py
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ==============================
# 3. PATHS
# ==============================
BASE_PATH = Path("/content/drive/MyDrive/things_fmri_project")

# persistent outputs only
PROC_PATH = BASE_PATH / "processed_trials"

# ephemeral raw storage (NEW)
RAW_TMP = Path("/content/raw_tmp")
RAW_TMP.mkdir(parents=True, exist_ok=True)


PROC_PATH.mkdir(parents=True, exist_ok=True)

# local temporary folder for large image-level arrays
LOCAL_TMP = Path("/content/image_level_tmp")
LOCAL_TMP.mkdir(parents=True, exist_ok=True)

# ==============================
# 4. SETTINGS
# ==============================
# Start with one subject first
subs = [1, 2, 3]

# Keep False at first. Turn True only after subject 1 works.
COPY_TO_DRIVE = False

# Feature chunk size for large H5 reading
CHUNK_SIZE = 2048

# ==============================
# 5. VALIDATION
# ==============================
def is_valid(sub_dir):
    stim_files = list(sub_dir.glob("*stimulus-metadata*.tsv"))
    h5_files = list(sub_dir.glob("*voxel-wise-responses*.h5"))
    return len(stim_files) > 0 and len(h5_files) > 0

# ==============================
# 6. DOWNLOAD SUBJECT DATA
# ==============================
def sync_subject(sub_str, target_dir):

    if target_dir.exists() and is_valid(target_dir):
        print(f"{sub_str} already downloaded. Skipping.")
        return

    if target_dir.exists():
        shutil.rmtree(target_dir)

    target_dir.mkdir(parents=True, exist_ok=True)

    s3_path = f"s3://openneuro.org/ds004192/derivatives/ICA-betas/{sub_str}/voxel-metadata/"

    print(f"\nSyncing {sub_str}...")

    result = subprocess.run([
        "aws", "s3", "sync",
        s3_path,
        str(target_dir),
        "--no-sign-request"
    ])

    if result.returncode != 0:
        raise RuntimeError(f"S3 sync failed for {sub_str}")

    print(f"Download complete: {sub_str}")

# ==============================
# 7. RUN DOWNLOAD
# ==============================
for sub in subs:
    sub_str = f"sub-0{sub}"
    target_dir = RAW_TMP / sub_str / "voxel-metadata"
    sync_subject(sub_str, target_dir)

# ==============================
# 8. SAFE METADATA LOADER
# ==============================
def load_metadata(path):
    # try TSV first
    df = pd.read_csv(path, sep="\t")

    # fallback if needed
    if len(df.columns) == 1:
        df = pd.read_csv(path, sep=",")

    print("Metadata columns:", list(df.columns))
    return df

# ==============================
# 9. BUILD TRUE IMAGE-LEVEL DATA
# ==============================
saved_output_paths = {}

for sub in subs:

    sub_str = f"sub-0{sub}"
    save_dir = PROC_PATH / sub_str
    save_dir.mkdir(parents=True, exist_ok=True)

    print("\n==============================")
    print(f"PROCESSING {sub_str}")
    print("==============================")

    raw_dir = RAW_TMP / sub_str / "voxel-metadata"

    stim_path = list(raw_dir.glob("*stimulus-metadata*.tsv"))[0]
    h5_path = list(raw_dir.glob("*voxel-wise-responses*.h5"))[0]

    # ---- LOAD METADATA ----
    stim_meta = load_metadata(stim_path)

    required_cols = ["trial_id", "stimulus", "concept"]
    for c in required_cols:
        if c not in stim_meta.columns:
            raise RuntimeError(f"Missing column: {c}")

    stim_meta = stim_meta.reset_index(drop=True)
    n_trials = len(stim_meta)

    # Save full metadata
    stim_meta.to_csv(save_dir / "stimulus_metadata.csv", index=False)

    # Trial index table
    trial_index = stim_meta[["trial_id", "stimulus", "concept"]].copy()
    trial_index["trial_col"] = np.arange(n_trials)
    trial_index.to_csv(save_dir / "trial_index.csv", index=False)

    # ---- BUILD IMAGE GROUPING ----
    # Keep stimulus order exactly as it appears in metadata
    image_ids, unique_stimuli = pd.factorize(trial_index["stimulus"], sort=False)
    n_images = len(unique_stimuli)

    # safety check: one exact stimulus should map to one concept
    concept_check = trial_index.groupby("stimulus", sort=False)["concept"].nunique()
    if (concept_check > 1).any():
        bad_stimulus = concept_check[concept_check > 1].index[0]
        raise RuntimeError(f"Stimulus {bad_stimulus} maps to multiple concepts.")

    image_counts = np.bincount(image_ids, minlength=n_images).astype(np.int32)

    image_index = (
        trial_index.groupby("stimulus", sort=False)
        .agg(
            concept=("concept", "first"),
            n_presentations=("stimulus", "size")
        )
        .reset_index()
    )
    image_index["image_row"] = np.arange(n_images)
    image_index.to_csv(save_dir / "image_index.csv", index=False)

    print("Metadata rows (trials):", n_trials)
    print("Unique stimuli:", n_images)

    # Sparse averaging matrix:
    # each trial contributes 1 / number_of_presentations to its image
    weights = (1.0 / image_counts[image_ids]).astype(np.float32)
    A = csr_matrix(
        (weights, (np.arange(n_trials), image_ids)),
        shape=(n_trials, n_images),
        dtype=np.float32
    )

    # ---- OPEN H5 AND DETECT ORIENTATION ----
    with h5py.File(h5_path, "r") as f:
        Y_raw = f["ResponseData"]["block0_values"]
        raw_shape = Y_raw.shape

        print("Raw H5 shape:", raw_shape)

        if raw_shape[1] == n_trials:
            orientation = "features_by_trials"
            n_features = raw_shape[0]
        elif raw_shape[0] == n_trials:
            orientation = "trials_by_features"
            n_features = raw_shape[1]
        else:
            raise RuntimeError(
                f"Could not match metadata rows ({n_trials}) to H5 shape {raw_shape} for {sub_str}."
            )

        print("Detected orientation:", orientation)
        print("Feature dimension:", n_features)

        # ---- SAVE LARGE OUTPUT LOCALLY FIRST ----
        out_path_local = LOCAL_TMP / f"Y_{sub_str}_images.npy"

        Y_image = np.lib.format.open_memmap(
            out_path_local,
            mode="w+",
            dtype=np.float32,
            shape=(n_images, n_features)
        )

        n_chunks = (n_features + CHUNK_SIZE - 1) // CHUNK_SIZE

        if orientation == "features_by_trials":
            # Y_raw shape = [features, trials]
            for chunk_idx, start in enumerate(range(0, n_features, CHUNK_SIZE)):
                end = min(start + CHUNK_SIZE, n_features)

                # block shape: [feature_chunk, trials]
                block = np.asarray(Y_raw[start:end, :], dtype=np.float32)

                # average repeated identical-image columns
                # A.T shape = [n_images, trials]
                # block.T shape = [trials, feature_chunk]
                # result = [n_images, feature_chunk]
                block_img = A.T @ block.T

                Y_image[:, start:end] = np.asarray(block_img, dtype=np.float32)

                if chunk_idx % 20 == 0 or end == n_features:
                    print(f"Processed chunk {chunk_idx + 1}/{n_chunks} | rows {start}:{end}")

        else:
            # Y_raw shape = [trials, features]
            for chunk_idx, start in enumerate(range(0, n_features, CHUNK_SIZE)):
                end = min(start + CHUNK_SIZE, n_features)

                # block shape: [trials, feature_chunk]
                block = np.asarray(Y_raw[:, start:end], dtype=np.float32)

                # A.T @ block -> [n_images, feature_chunk]
                block_img = A.T @ block

                Y_image[:, start:end] = np.asarray(block_img, dtype=np.float32)

                if chunk_idx % 20 == 0 or end == n_features:
                    print(f"Processed chunk {chunk_idx + 1}/{n_chunks} | cols {start}:{end}")

        # flush local file
        del Y_image

    # ---- OPTIONAL COPY TO DRIVE ----
    if COPY_TO_DRIVE:
        out_path_drive = save_dir / f"Y_{sub_str}_images.npy"
        shutil.copy2(out_path_local, out_path_drive)
        saved_output_paths[sub_str] = out_path_drive
        print(f"Copied image-level dataset to Drive: {out_path_drive}")
    else:
        saved_output_paths[sub_str] = out_path_local
        print(f"Saved image-level dataset locally: {out_path_local}")

# optional but recommended cleanup
shutil.rmtree(RAW_TMP, ignore_errors=True)
RAW_TMP.mkdir(parents=True, exist_ok=True)

# ==============================
# 10. FINAL CHECK
# ==============================
print("\n===== FINAL CHECK =====")

for sub in subs:
    sub_str = f"sub-0{sub}"
    save_dir = PROC_PATH / sub_str

    image_meta = pd.read_csv(save_dir / "image_index.csv")
    y_path = saved_output_paths[sub_str]
    Y_image = np.load(y_path, mmap_mode="r")

    print(f"\n{sub_str}")
    print("Image-level Y path:", y_path)
    print("Image-level Y shape:", Y_image.shape)
    print("Image metadata shape:", image_meta.shape)
    print("Stimulus examples:", image_meta["stimulus"].head().tolist())
    print("Concept examples:", image_meta["concept"].head().tolist())
    print("Presentation counts:", image_meta["n_presentations"].head().tolist())

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.

Syncing sub-01...
Download complete: sub-01

Syncing sub-02...
Download complete: sub-02

Syncing sub-03...
Download complete: sub-03

PROCESSING sub-01
Metadata columns: ['trial_type', 'session', 'run', 'subject_id', 'trial_id', 'stimulus', 'concept']
Metadata rows (trials): 9840
Unique stimuli: 8740
Raw H5 shape: (211339, 9840)
Detected orientation: features_by_trials
Feature dimensio

In [11]:
# ==========================================
# CHECK A: LEAVE-ONE-OUT WITHIN vs BETWEEN
# small random sample of concepts
# ==========================================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# ----- SETTINGS -----
sub = "sub-01"
n_concepts_sample = 80     # small random sample of concepts
n_features_sample = 1500   # feature subset for speed
np.random.seed(42)

# ----- PATHS -----
BASE = Path("/content/drive/MyDrive/things_fmri_project/processed_trials") / sub
meta = pd.read_csv(BASE / "image_index.csv")
Y = np.load(f"/content/image_level_tmp/Y_{sub}_images.npy", mmap_mode="r")

# ----- SAMPLE CONCEPTS -----
all_concepts = meta["concept"].unique()
sampled_concepts = np.random.choice(
    all_concepts, size=min(n_concepts_sample, len(all_concepts)), replace=False
)

mask = meta["concept"].isin(sampled_concepts).values
meta_sub = meta[mask].reset_index(drop=True)

# sample features for speed
feat_idx = np.sort(np.random.choice(Y.shape[1], size=n_features_sample, replace=False))
Y_sub = np.asarray(Y[mask][:, feat_idx], dtype=np.float32)

labels = meta_sub["concept"].values
concepts = np.unique(labels)

within_scores = []
between_scores = []

for c in concepts:
    idx_c = np.where(labels == c)[0]
    if len(idx_c) < 2:
        continue

    other_concepts = concepts[concepts != c]
    other_c = np.random.choice(other_concepts)
    idx_other = np.where(labels == other_c)[0]

    for i in idx_c:
        # leave-one-out prototype for same concept
        idx_loo = idx_c[idx_c != i]
        if len(idx_loo) == 0:
            continue

        proto_within = Y_sub[idx_loo].mean(axis=0, keepdims=True)
        sim_within = cosine_similarity(Y_sub[i:i+1], proto_within)[0, 0]
        within_scores.append(sim_within)

        # random other concept prototype
        proto_between = Y_sub[idx_other].mean(axis=0, keepdims=True)
        sim_between = cosine_similarity(Y_sub[i:i+1], proto_between)[0, 0]
        between_scores.append(sim_between)

print("Sampled concepts:", len(concepts))
print("Mean leave-one-out within-concept similarity :", float(np.mean(within_scores)))
print("Mean between-concept similarity              :", float(np.mean(between_scores)))
print("Gap (within - between)                       :", float(np.mean(within_scores) - np.mean(between_scores)))

Sampled concepts: 80
Mean leave-one-out within-concept similarity : 0.06889760494232178
Mean between-concept similarity              : 0.06653989106416702
Gap (within - between)                       : 0.0023577138781547546


In [17]:
# ==============================
# 0. MOUNT DRIVE
# ==============================
from google.colab import drive
drive.mount('/content/drive')

# ==============================
# 1. IMPORTS
# ==============================
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ==============================
# 2. PATHS (EDIT THESE)
# ==============================

# metadata (already aligned with your Y)
META_PATH = Path("/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/stimulus_metadata.csv")

# image roots (UPLOAD THESE TO DRIVE FIRST)
IMG_ROOT_1 = Path("/content/drive/MyDrive/things_1half")
IMG_ROOT_2 = Path("/content/drive/MyDrive/things_2half")

SAVE_PATH = Path("/content/drive/MyDrive/things_fmri_project/embeddings")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

# ==============================
# 3. LOAD METADATA
# ==============================
meta = pd.read_csv(META_PATH)

stimuli = meta["stimulus"].tolist()

print("Total trials:", len(stimuli))

# ==============================
# 4. LOAD CLIP
# ==============================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

model.eval()

# ==============================
# 5. HELPER: FIND IMAGE PATH
# ==============================
def find_image(stimulus):
    concept = stimulus.split("_")[0]

    p1 = IMG_ROOT_1 / concept / stimulus
    if p1.exists():
        return p1

    p2 = IMG_ROOT_2 / concept / stimulus
    if p2.exists():
        return p2

    return None

# ==============================
# 6. COMPUTE EMBEDDINGS
# ==============================
embeddings = []
missing = []

for stim in tqdm(stimuli):

    img_path = find_image(stim)

    if img_path is None:
        missing.append(stim)
        embeddings.append(np.zeros(512))
        continue

    image = Image.open(img_path).convert("RGB")

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.get_image_features(**inputs)

    # handle weird return types
    if hasattr(outputs, "pooler_output"):
        emb = outputs.pooler_output
    else:
        emb = outputs

    # normalize
    emb = emb / emb.norm(dim=-1, keepdim=True)

    emb = emb.cpu().numpy().squeeze()

    embeddings.append(emb)

# ==============================
# 7. FINAL MATRIX
# ==============================
X = np.vstack(embeddings)

print("Embedding matrix shape:", X.shape)

# ==============================
# 8. SAVE
# ==============================
np.save(SAVE_PATH / "X_clip_embeddings.npy", X)

pd.Series(missing).to_csv(SAVE_PATH / "missing_images.csv", index=False)

print("Missing images:", len(missing))
print("Saved embeddings.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total trials: 9840


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 9840/9840 [2:37:11<00:00,  1.04it/s]


Embedding matrix shape: (9840, 512)
Missing images: 792
Saved embeddings.


In [32]:
import numpy as np
import pandas as pd

# --------------------------
# LOAD
# --------------------------
X = np.load("/content/drive/MyDrive/things_fmri_project/embeddings/X_clip_embeddings.npy")

meta = pd.read_csv(
    "/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/stimulus_metadata.csv"
)

missing = pd.read_csv(
    "/content/drive/MyDrive/things_fmri_project/embeddings/missing_images.csv"
)

stimulus = meta["stimulus"].values
missing_set = set(missing.iloc[:, 0].values)

# --------------------------
# STEP 1: BUILD STABLE IMAGE ORDER (NO np.unique)
# --------------------------
seen = {}
image_index = []
stim_ids = np.zeros(len(stimulus), dtype=int)

for i, s in enumerate(stimulus):
    if s not in seen:
        seen[s] = len(seen)
        image_index.append(s)
    stim_ids[i] = seen[s]

image_index = np.array(image_index)

n_images = len(image_index)
dim = X.shape[1]

print("Unique images (pre-filter):", n_images)

# --------------------------
# STEP 2: AGGREGATE TRIALS → IMAGE SPACE
# --------------------------
X_sum = np.zeros((n_images, dim), dtype=np.float32)
counts = np.zeros(n_images, dtype=np.int32)

for i in range(len(X)):
    idx = stim_ids[i]
    X_sum[idx] += X[i]
    counts[idx] += 1

X_image = X_sum / np.maximum(counts[:, None], 1)

# --------------------------
# STEP 3: BUILD VALID MASK (8003 definition)
# --------------------------
valid_mask = np.array([
    s not in missing_set for s in image_index
])

print("Valid images:", valid_mask.sum())

# --------------------------
# STEP 4: FINAL ALIGNMENT
# --------------------------
X_clip_aligned = X_image[valid_mask]

print("Final shape:", X_clip_aligned.shape)

np.save(
    "/content/drive/MyDrive/things_fmri_project/embeddings/X_clip_aligned.npy",
    X_clip_aligned
)

# OPTIONAL: save stimulus order for locking alignment
np.save(
    "/content/drive/MyDrive/things_fmri_project/embeddings/clip_stimulus_8003.npy",
    image_index[valid_mask]
)

Unique images (pre-filter): 8740
Valid images: 8003
Final shape: (8003, 512)


In [33]:
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project/embeddings")

# rebuild same logic (must match your run exactly)
meta = pd.read_csv(
    "/content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/stimulus_metadata.csv"
)

missing = pd.read_csv(BASE / "missing_images.csv")
missing_set = set(missing.iloc[:, 0].values)

stimulus = meta["stimulus"].values

seen = {}
image_index = []
for s in stimulus:
    if s not in seen:
        seen[s] = len(seen)
        image_index.append(s)

image_index = np.array(image_index)

valid_mask = np.array([s not in missing_set for s in image_index])

stimulus_8003 = image_index[valid_mask]

print("Final stimulus count:", len(stimulus_8003))

np.save(BASE / "stimulus_8003.npy", stimulus_8003)

Final stimulus count: 8003


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def align_subject_Y_chunked(sub, stimulus_8003, chunk_size=256):

    base = Path("/content/drive/MyDrive/things_fmri_project/processed_trials") / sub

    Y_path = Path("/content/image_level_tmp") / f"Y_{sub}_images.npy"
    idx_path = base / "image_index.csv"

    print(f"\n==============================")
    print(f"CHUNK ALIGNING {sub}")
    print(f"==============================")

    Y = np.load(Y_path, mmap_mode="r")
    idx = pd.read_csv(idx_path)

    stim_to_row = {s: i for i, s in enumerate(idx["stimulus"].values)}

    n = len(stimulus_8003)
    d = Y.shape[1]

    out_path = base / f"Y_{sub}_aligned.npy"

    # create memmap output (CRITICAL FIX)
    Y_aligned = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(n, d)
    )

    missing = 0
    used = 0

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)

        chunk_stims = stimulus_8003[start:end]

        for i, s in enumerate(chunk_stims, start=start):
            row = stim_to_row.get(s, -1)

            if row != -1:
                Y_aligned[i] = Y[row]
                used += 1
            else:
                missing += 1

        if start % (chunk_size * 5) == 0:
            print(f"{sub}: processed {end}/{n}")

    print("\nDONE:", sub)
    print("Used:", used)
    print("Missing:", missing)
    print("Saved to:", out_path)

    return out_path

In [2]:
stimulus_8003 = np.load(
    "/content/drive/MyDrive/things_fmri_project/embeddings/stimulus_8003.npy",
    allow_pickle=True
)

for sub in ["sub-01", "sub-02", "sub-03"]:
    align_subject_Y_chunked(sub, stimulus_8003)

print("ALL SUBJECTS ALIGNED (SAFE MODE) ✅")


CHUNK ALIGNING sub-01
sub-01: processed 256/8003
sub-01: processed 1536/8003
sub-01: processed 2816/8003
sub-01: processed 4096/8003
sub-01: processed 5376/8003
sub-01: processed 6656/8003
sub-01: processed 7936/8003

DONE: sub-01
Used: 8003
Missing: 0
Saved to: /content/drive/MyDrive/things_fmri_project/processed_trials/sub-01/Y_sub-01_aligned.npy

CHUNK ALIGNING sub-02
sub-02: processed 256/8003
sub-02: processed 1536/8003
sub-02: processed 2816/8003
sub-02: processed 4096/8003
sub-02: processed 5376/8003
sub-02: processed 6656/8003
sub-02: processed 7936/8003

DONE: sub-02
Used: 8003
Missing: 0
Saved to: /content/drive/MyDrive/things_fmri_project/processed_trials/sub-02/Y_sub-02_aligned.npy

CHUNK ALIGNING sub-03
sub-03: processed 256/8003
sub-03: processed 1536/8003
sub-03: processed 2816/8003
sub-03: processed 4096/8003
sub-03: processed 5376/8003
sub-03: processed 6656/8003
sub-03: processed 7936/8003

DONE: sub-03
Used: 8003
Missing: 0
Saved to: /content/drive/MyDrive/things_fm

order the y data

In [13]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project/processed_trials")

for sub in ["sub-01", "sub-02", "sub-03"]:
    print("\n", sub)
    p = BASE / sub
    for f in p.glob("*.npy"):
        print("  ", f.name)


 sub-01

 sub-02
   Y_train_scaled.npy
   Y_test_scaled.npy

 sub-03
   Y_train_scaled.npy
   Y_test_scaled.npy


In [11]:
from pathlib import Path
import numpy as np

# ==============================
# CONFIG
# ==============================
BASE_PROC = Path("/content/drive/MyDrive/things_fmri_project/processed_trials")
BASE_EMB = Path("/content/drive/MyDrive/things_fmri_project/embeddings")

subjects = ["sub-01"]#, "sub-02", "sub-03"]

# ==============================
# LOAD X (ONLY FOR SPLIT)
# ==============================
X = np.load(BASE_EMB / "X_clip_aligned.npy", mmap_mode="r")
N = X.shape[0]

print("Total samples:", N)

# ==============================
# SHARED TRAIN / TEST SPLIT
# ==============================
np.random.seed(42)

indices = np.arange(N)
np.random.shuffle(indices)

split = int(0.75 * N)

train_idx = indices[:split]
test_idx = indices[split:]

print("Train size:", len(train_idx))
print("Test size:", len(test_idx))

# Save split ONCE (important for reproducibility)
np.save(BASE_EMB / "train_indices.npy", train_idx)
np.save(BASE_EMB / "test_indices.npy", test_idx)

# ==============================
# PROCESS EACH SUBJECT
# ==============================
chunk_size = 256

for sub in subjects:

    print(f"\n==============================")
    print(f"SCALING {sub}")
    print(f"==============================")

    Y_path = BASE_PROC / sub / f"Y_{sub}_aligned.npy"
    Y = np.load(Y_path, mmap_mode="r")

    d = Y.shape[1]

    # --------------------------
    # COMPUTE MEAN
    # --------------------------
    mean = np.zeros(d, dtype=np.float64)
    count = 0

    for start in range(0, len(train_idx), chunk_size):
        batch_idx = train_idx[start:start+chunk_size]
        batch = Y[batch_idx]

        mean += batch.sum(axis=0)
        count += batch.shape[0]

    mean /= count

    print("Mean done.")

    # --------------------------
    # COMPUTE STD
    # --------------------------
    var = np.zeros(d, dtype=np.float64)

    for start in range(0, len(train_idx), chunk_size):
        batch_idx = train_idx[start:start+chunk_size]
        batch = Y[batch_idx]

        var += ((batch - mean) ** 2).sum(axis=0)

    var /= count
    std = np.sqrt(var)

    std[std == 0] = 1.0

    print("Std done.")

    # --------------------------
    # SCALE FUNCTION
    # --------------------------
    def scale(indices, out_path):

        out = np.lib.format.open_memmap(
            out_path,
            mode="w+",
            dtype=np.float32,
            shape=(len(indices), d)
        )

        for start in range(0, len(indices), chunk_size):
            batch_idx = indices[start:start+chunk_size]
            batch = Y[batch_idx]

            out[start:start+len(batch)] = (batch - mean) / std

        return out

    # --------------------------
    # APPLY SCALING
    # --------------------------
    scale(train_idx, BASE_PROC / sub / "Y_train_scaled.npy")
    scale(test_idx,  BASE_PROC / sub / "Y_test_scaled.npy")

    print(f"{sub} COMPLETE")

print("\nALL SUBJECTS SCALED ✅")

Total samples: 8003
Train size: 6002
Test size: 2001

SCALING sub-01
Mean done.
Std done.
sub-01 COMPLETE

SCALING sub-02
Mean done.
Std done.
sub-02 COMPLETE

SCALING sub-03
Mean done.
Std done.
sub-03 COMPLETE

ALL SUBJECTS SCALED ✅


In [14]:
from pathlib import Path
import shutil

BASE_PROC = Path("/content/drive/MyDrive/things_fmri_project/processed_trials")
OUT_DIR = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

subjects = ["sub-01", "sub-02", "sub-03"]

for sub in subjects:

    print(f"\nProcessing {sub}")

    sub_dir = BASE_PROC / sub

    train_candidates = [
        sub_dir / "Y_train_scaled.npy",
        sub_dir / f"Y_{sub}_train.npy",
    ]

    test_candidates = [
        sub_dir / "Y_test_scaled.npy",
        sub_dir / f"Y_{sub}_test.npy",
    ]

    train_src = next((p for p in train_candidates if p.exists()), None)
    test_src  = next((p for p in test_candidates if p.exists()), None)

    if train_src is None or test_src is None:
        print(f"⚠️ Missing files for {sub}")
        continue

    shutil.copy(train_src, OUT_DIR / f"Y_{sub}_train.npy")
    shutil.copy(test_src,  OUT_DIR / f"Y_{sub}_test.npy")

    print(f"{sub} exported safely")

print("\nDONE EXPORTING CLEANLY ✅")


Processing sub-01
⚠️ Missing files for sub-01

Processing sub-02
sub-02 exported safely

Processing sub-03
sub-03 exported safely

DONE EXPORTING CLEANLY ✅


run models

In [18]:
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy")
X_test  = np.load(BASE / "X_test.npy")

print(X_train.shape, X_test.shape)

Y_train = np.load(BASE / "Y_sub-01_train.npy", mmap_mode="r")
Y_test  = np.load(BASE / "Y_sub-01_test.npy", mmap_mode="r")

print(Y_train.shape, Y_test.shape)

(6002, 512) (2001, 512)
(6002, 211339) (2001, 211339)


In [22]:
import numpy as np
from pathlib import Path
from sklearn.linear_model import Ridge
from tqdm import tqdm

# ==============================
# LOAD DATA
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy")
X_test  = np.load(BASE / "X_test.npy")

Y_train = np.load(BASE / "Y_sub-03_train.npy", mmap_mode="r")
Y_test  = np.load(BASE / "Y_sub-03_test.npy", mmap_mode="r")

print("X:", X_train.shape, X_test.shape)
print("Y:", Y_train.shape, Y_test.shape)

# ==============================
# SETTINGS
# ==============================
alpha = 100.0           # ridge strength (can tune later)
voxel_chunk = 5000      # safe chunk size

n_voxels = Y_train.shape[1]
n_test = Y_test.shape[0]

Y_pred = np.zeros((n_test, n_voxels), dtype=np.float32)

# ==============================
# RIDGE FIT PER VOXEL CHUNK
# ==============================
print("\nTraining ridge (voxel chunks)...")

for start in tqdm(range(0, n_voxels, voxel_chunk)):
    end = min(start + voxel_chunk, n_voxels)

    Y_chunk = Y_train[:, start:end]

    model = Ridge(alpha=alpha, fit_intercept=True)

    model.fit(X_train, Y_chunk)

    Y_pred[:, start:end] = model.predict(X_test)

print("\nComputing full evaluation metrics...")

# ==============================
# R² (voxel-wise, stable)
# ==============================
y_mean = Y_test.mean(axis=0)

ss_tot = np.sum((Y_test - y_mean) ** 2, axis=0)
ss_res = np.sum((Y_test - Y_pred) ** 2, axis=0)

# avoid divide-by-zero issues
r2 = np.ones_like(ss_tot)
valid = ss_tot > 1e-12
r2[valid] = 1 - (ss_res[valid] / ss_tot[valid])

# ==============================
# MSE (voxel-wise)
# ==============================
mse_vox = np.mean((Y_test - Y_pred) ** 2, axis=0)

# ==============================
# MAE (voxel-wise)
# ==============================
mae_vox = np.mean(np.abs(Y_test - Y_pred), axis=0)

# ==============================
# GLOBAL METRICS
# ==============================
mse_global = float(np.mean((Y_test - Y_pred) ** 2))
mae_global = float(np.mean(np.abs(Y_test - Y_pred)))

# ==============================
# SUMMARY
# ==============================
print("\n===== RIDGE PERFORMANCE (SUB-03) =====")

print("\nR²:")
print("  Mean:", float(np.mean(r2)))
print("  Median:", float(np.median(r2)))
print("  Top 1%:", float(np.mean(np.sort(r2)[-max(1, int(0.01 * len(r2))):])))
print("  Negative voxels:", float(np.mean(r2 < 0)))

print("\nMSE (voxel-wise):")
print("  Mean:", float(np.mean(mse_vox)))
print("  Median:", float(np.median(mse_vox)))

print("\nMAE (voxel-wise):")
print("  Mean:", float(np.mean(mae_vox)))
print("  Median:", float(np.median(mae_vox)))

print("\n===== GLOBAL =====")
print("MSE:", mse_global)
print("MAE:", mae_global)

# ==============================
# SAVE RESULTS
# ==============================
np.save(BASE / "sub-03_ridge_predictions.npy", Y_pred)
np.save(BASE / "sub-03_r2.npy", r2)
np.save(BASE / "sub-03_mse.npy", mse_vox)
np.save(BASE / "sub-03_mae.npy", mae_vox)

print("\nDONE ✅")

X: (6002, 512) (2001, 512)
Y: (6002, 189164) (2001, 189164)

Training ridge (voxel chunks)...


100%|██████████| 38/38 [03:42<00:00,  5.85s/it]



Computing full evaluation metrics...

===== RIDGE PERFORMANCE (SUB-03) =====

R²:
  Mean: -0.0009077139548026025
  Median: -0.0009740591049194336
  Top 1%: 0.0128642488270998
  Negative voxels: 0.7646698103233174

MSE (voxel-wise):
  Mean: 1.0007073879241943
  Median: 0.9999234676361084

MAE (voxel-wise):
  Mean: 0.7889111042022705
  Median: 0.7930705547332764

===== GLOBAL =====
MSE: 1.0007116794586182
MAE: 0.7889131307601929

DONE ✅


In [23]:
import numpy as np
from pathlib import Path
from sklearn.linear_model import Lasso
from tqdm import tqdm

# ==============================
# LOAD DATA
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy")
X_test  = np.load(BASE / "X_test.npy")

Y_train = np.load(BASE / "Y_sub-01_train.npy", mmap_mode="r")
Y_test  = np.load(BASE / "Y_sub-01_test.npy", mmap_mode="r")

print("X:", X_train.shape, X_test.shape)
print("Y:", Y_train.shape, Y_test.shape)

# ==============================
# SETTINGS
# ==============================
alpha = 0.1              # Lasso strength (IMPORTANT: lower than Ridge scale)
voxel_chunk = 2000       # smaller chunk (Lasso is heavier)

n_voxels = Y_train.shape[1]
n_test = Y_test.shape[0]

Y_pred = np.zeros((n_test, n_voxels), dtype=np.float32)

# ==============================
# LASSO FIT PER VOXEL CHUNK
# ==============================
print("\nTraining Lasso (voxel chunks)...")

for start in tqdm(range(0, n_voxels, voxel_chunk)):
    end = min(start + voxel_chunk, n_voxels)

    Y_chunk = Y_train[:, start:end]

    model = Lasso(
        alpha=alpha,
        fit_intercept=True,
        max_iter=2000,
        tol=1e-3
    )

    model.fit(X_train, Y_chunk)

    Y_pred[:, start:end] = model.predict(X_test)

# ==============================
# METRICS
# ==============================
print("\nComputing metrics...")

# R²
y_mean = Y_test.mean(axis=0)
ss_tot = np.sum((Y_test - y_mean) ** 2, axis=0)
ss_res = np.sum((Y_test - Y_pred) ** 2, axis=0)

r2 = 1 - (ss_res / ss_tot)

# MSE
mse_vox = np.mean((Y_test - Y_pred) ** 2, axis=0)

# MAE
mae_vox = np.mean(np.abs(Y_test - Y_pred), axis=0)

# ==============================
# SUMMARY
# ==============================
print("\n===== LASSO PERFORMANCE (SUB-01) =====")

print("\nR²:")
print("  Mean:", float(np.mean(r2)))
print("  Median:", float(np.median(r2)))
print("  Top 1%:", float(np.mean(np.sort(r2)[-max(1, int(0.01 * len(r2))):])))
print("  Negative voxels:", float(np.mean(r2 < 0)))

print("\nMSE:")
print("  Mean:", float(np.mean(mse_vox)))

print("\nMAE:")
print("  Mean:", float(np.mean(mae_vox)))

# ==============================
# SAVE
# ==============================
np.save(BASE / "sub-01_lasso_predictions.npy", Y_pred)
np.save(BASE / "sub-01_lasso_r2.npy", r2)

print("\nDONE ✅")

X: (6002, 512) (2001, 512)
Y: (6002, 211339) (2001, 211339)

Training Lasso (voxel chunks)...


100%|██████████| 106/106 [1:14:34<00:00, 42.21s/it]



Computing metrics...

===== LASSO PERFORMANCE (SUB-01) =====

R²:
  Mean: -0.0006884154863655567
  Median: -0.0003167390823364258
  Top 1%: 7.840281455173681e-07
  Negative voxels: 0.9872810981409016

MSE:
  Mean: 1.0056320428848267

MAE:
  Mean: 0.793121337890625

DONE ✅


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Run PCA

In [1]:
import numpy as np
from pathlib import Path
from sklearn.decomposition import IncrementalPCA

# ==============================
# LOAD DATA (MEMMAP SAFE)
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

Y_train = np.load(BASE / "Y_sub-03_train.npy", mmap_mode="r").astype(np.float32)
Y_test  = np.load(BASE / "Y_sub-03_test.npy", mmap_mode="r").astype(np.float32)

print("Train:", Y_train.shape)
print("Test:", Y_test.shape)

# ==============================
# SETTINGS
# ==============================
n_components = 200      # you can try 100, 200, 300
batch_size = 200        # memory-safe chunks

# ==============================
# FIT PCA (INCREMENTAL)
# ==============================
print("\nFitting Incremental PCA...")

ipca = IncrementalPCA(n_components=n_components, batch_size=batch_size)

# fit in chunks (CRITICAL for RAM safety)
for start in range(0, Y_train.shape[0], batch_size):
    end = min(start + batch_size, Y_train.shape[0])
    ipca.partial_fit(Y_train[start:end])

print("PCA fitted.")

# ==============================
# TRANSFORM
# ==============================
print("Transforming train/test...")

Y_train_pca = np.zeros((Y_train.shape[0], n_components), dtype=np.float32)
Y_test_pca  = np.zeros((Y_test.shape[0], n_components), dtype=np.float32)

for start in range(0, Y_train.shape[0], batch_size):
    end = min(start + batch_size, Y_train.shape[0])
    Y_train_pca[start:end] = ipca.transform(Y_train[start:end])

for start in range(0, Y_test.shape[0], batch_size):
    end = min(start + batch_size, Y_test.shape[0])
    Y_test_pca[start:end] = ipca.transform(Y_test[start:end])

print("Done.")

# ==============================
# SAVE
# ==============================
np.save(BASE / "sub-03_Y_train_pca.npy", Y_train_pca)
np.save(BASE / "sub-03_Y_test_pca.npy", Y_test_pca)

np.save(BASE / "sub-03_pca_model_components.npy", ipca.components_)
np.save(BASE / "sub-03_pca_model_mean.npy", ipca.mean_)

print("\nSAVED ✅")
print("Output shape:", Y_train_pca.shape)

Train: (6002, 189164)
Test: (2001, 189164)

Fitting Incremental PCA...
PCA fitted.
Transforming train/test...
Done.

SAVED ✅
Output shape: (6002, 200)


# Run PLS

In [30]:
import numpy as np
from pathlib import Path
from sklearn.cross_decomposition import PLSRegression

# ==============================
# LOAD DATA (PCA SPACE)
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy").astype(np.float32)
X_test  = np.load(BASE / "X_test.npy").astype(np.float32)

# IMPORTANT: use PCA outputs
Y_train = np.load(BASE / "sub-03_Y_train_pca.npy")
Y_test  = np.load(BASE / "sub-03_Y_test_pca.npy")

print("Shapes:")
print(X_train.shape, Y_train.shape)

# ==============================
# CLEAN MEMORY LAYOUT
# ==============================
X_train = np.ascontiguousarray(X_train)
X_test  = np.ascontiguousarray(X_test)

# ==============================
# PLS
# ==============================
n_components = 50  # can try 10–50

pls = PLSRegression(
    n_components=n_components,
    scale=False,
    max_iter=500
)

print("\nFitting PLS...")

pls.fit(X_train, Y_train)

print("Transforming...")

Y_train_pls = pls.transform(X_train)
Y_test_pls  = pls.transform(X_test)

print("Output shapes:")
print(Y_train_pls.shape, Y_test_pls.shape)

# ==============================
# SAVE
# ==============================
np.save(BASE / "sub-03_Y_train_pls.npy", Y_train_pls.astype(np.float32))
np.save(BASE / "sub-03_Y_test_pls.npy", Y_test_pls.astype(np.float32))

np.save(BASE / "sub-03_pls_x_weights.npy", pls.x_weights_)
np.save(BASE / "sub-03_pls_y_weights.npy", pls.y_weights_)
np.save(BASE / "sub-03_pls_y_loadings.npy", pls.y_loadings_)

print("\nDONE ✅")

Shapes:
(6002, 512) (6002, 200)

Fitting PLS...
Transforming...
Output shapes:
(6002, 50) (2001, 50)

DONE ✅


# Run OLS, Ridge, Lasso (change files for subjects and PCA/PLS data

In [31]:
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

# ==============================
# LOAD DATA
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy").astype(np.float32)
X_test  = np.load(BASE / "X_test.npy").astype(np.float32)

Y_train = np.load(BASE / "sub-03_Y_train_pls.npy").astype(np.float32)
Y_test  = np.load(BASE / "sub-03_Y_test_pls.npy").astype(np.float32)

print("Shapes:", X_train.shape, Y_train.shape)

# ==============================
# MODELS
# ==============================
ridge_alphas = [0.1, 1.0, 10.0]
lasso_alphas = [0.001, 0.01, 0.1]

results = []

# ==============================
# 1. OLS
# ==============================
print("\nRunning OLS...")

ols = LinearRegression()
ols.fit(X_train, Y_train)

Y_train_pred = ols.predict(X_train)
Y_test_pred  = ols.predict(X_test)

results.append((
    "OLS",
    r2_score(Y_train, Y_train_pred),
    r2_score(Y_test, Y_test_pred),
    mean_squared_error(Y_train, Y_train_pred),
    mean_squared_error(Y_test, Y_test_pred),
))

# ==============================
# 2. RIDGE
# ==============================
for alpha in ridge_alphas:
    print(f"\nRunning Ridge alpha={alpha}")

    model = Ridge(alpha=alpha)
    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    Y_test_pred  = model.predict(X_test)

    results.append((
        f"Ridge α={alpha}",
        r2_score(Y_train, Y_train_pred),
        r2_score(Y_test, Y_test_pred),
        mean_squared_error(Y_train, Y_train_pred),
        mean_squared_error(Y_test, Y_test_pred),
    ))

# ==============================
# 3. LASSO
# ==============================
for alpha in lasso_alphas:
    print(f"\nRunning Lasso alpha={alpha}")

    model = Lasso(alpha=alpha, max_iter=2000)

    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    Y_test_pred  = model.predict(X_test)

    results.append((
        f"Lasso α={alpha}",
        r2_score(Y_train, Y_train_pred),
        r2_score(Y_test, Y_test_pred),
        mean_squared_error(Y_train, Y_train_pred),
        mean_squared_error(Y_test, Y_test_pred),
    ))

# ==============================
# PRINT RESULTS
# ==============================
print("\n===== MODEL COMPARISON (SUB-03) =====\n")

print(f"{'Model':<15} {'R2 Train':>10} {'R2 Test':>10} {'MSE Train':>12} {'MSE Test':>12}")
print("-"*65)

for r in results:
    name, r2_tr, r2_te, mse_tr, mse_te = r
    print(f"{name:<15} {r2_tr:10.4f} {r2_te:10.4f} {mse_tr:12.4f} {mse_te:12.4f}")

print("\nDONE ✅")

Shapes: (6002, 512) (6002, 50)

Running OLS...

Running Ridge alpha=0.1

Running Ridge alpha=1.0

Running Ridge alpha=10.0

Running Lasso alpha=0.001

Running Lasso alpha=0.01

Running Lasso alpha=0.1

===== MODEL COMPARISON (SUB-03) =====

Model             R2 Train    R2 Test    MSE Train     MSE Test
-----------------------------------------------------------------
OLS                 1.0000     1.0000       0.0000       0.0000
Ridge α=0.1         0.9994     0.9993       0.0000       0.0000
Ridge α=1.0         0.9813     0.9798       0.0000       0.0000
Ridge α=10.0        0.8112     0.8062       0.0005       0.0005
Lasso α=0.001       0.0975     0.0969       0.0026       0.0025
Lasso α=0.01       -0.0000    -0.0004       0.0035       0.0035
Lasso α=0.1        -0.0000    -0.0004       0.0035       0.0035

DONE ✅


# Convert back to larger space (ONLY for PLS)

In [32]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")
sub = "sub-03"

chunk_size = 5000

# ==============================
# LOAD
# ==============================
Y_test_voxel = np.load(BASE / f"Y_{sub}_test.npy", mmap_mode="r")

Y_pred_pls   = np.load(BASE / f"{sub}_Y_test_pls.npy")  # (N, 20)

pca_components = np.load(BASE / f"{sub}_pca_model_components.npy")  # (200, voxels)
pca_mean       = np.load(BASE / f"{sub}_pca_model_mean.npy")        # (voxels,)

pls_y_loadings = np.load(BASE / f"{sub}_pls_y_loadings.npy")        # (200, 20)

print("Loaded.")

# ==============================
# STEP 1: PLS → PCA
# ==============================
Y_pred_pca = Y_pred_pls @ pls_y_loadings.T   # (N, 200)


# ==============================
# STEP 2: PCA → VOXEL (chunked)
# ==============================
n_samples, n_voxels = Y_test_voxel.shape

r2_all = []
mse_all = []

print("\nReconstructing + evaluating...")

for start in tqdm(range(0, n_voxels, chunk_size)):
    end = min(start + chunk_size, n_voxels)

    comp_chunk = pca_components[:, start:end]   # (200, chunk)

    Y_pred_chunk = Y_pred_pca @ comp_chunk + pca_mean[start:end]
    Y_true = Y_test_voxel[:, start:end]

    mse = np.mean((Y_true - Y_pred_chunk) ** 2, axis=0)

    ss_tot = np.sum((Y_true - Y_true.mean(axis=0)) ** 2, axis=0)
    ss_res = np.sum((Y_true - Y_pred_chunk) ** 2, axis=0)

    r2 = 1 - ss_res / np.maximum(ss_tot, 1e-8)

    r2_all.append(r2)
    mse_all.append(mse)

r2_all = np.concatenate(r2_all)
mse_all = np.concatenate(mse_all)

print("\n===== FINAL RESULTS =====")
print("R² mean:", float(np.mean(r2_all)))
print("Top 1%:", float(np.mean(np.sort(r2_all)[-int(0.01 * len(r2_all)):])))
print("Negative voxels:", float(np.mean(r2_all < 0)))
print("MSE mean:", float(np.mean(mse_all)))

print("\nDONE ✅")

Loaded.

Reconstructing + evaluating...


100%|██████████| 38/38 [00:53<00:00,  1.40s/it]


===== FINAL RESULTS =====
R² mean: -0.00849802395543575
Top 1%: 0.0076053455714841
Negative voxels: 0.9120445750777103
MSE mean: 1.0082295262868561

DONE ✅


# XGBoost

In [23]:
import numpy as np
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from tqdm import tqdm

# ==============================
# LOAD DATA
# ==============================
BASE = Path("/content/drive/MyDrive/things_fmri_project/final_dataset")

X_train = np.load(BASE / "X_train.npy").astype(np.float32)
X_test  = np.load(BASE / "X_test.npy").astype(np.float32)

Y_train = np.load(BASE / "sub-03_Y_train_pca.npy").astype(np.float32)
Y_test  = np.load(BASE / "sub-03_Y_test_pca.npy").astype(np.float32)

print("Shapes:", X_train.shape, Y_train.shape)

# ==============================
# SETTINGS (IMPORTANT FOR STABILITY)
# ==============================
params = {
    "n_estimators": 400,        # increase but...
    "learning_rate": 0.02,      # reduce
    "max_depth": 2,             # VERY important
    "min_child_weight": 10,     # suppress noisy splits
    "subsample": 0.5,           # more randomness
    "colsample_bytree": 0.5,    # reduce feature reliance
    "reg_alpha": 5.0,           # stronger L1
    "reg_lambda": 20.0,         # stronger L2
    "gamma": 1.0,               # require split gain
    "objective": "reg:squarederror",
    "tree_method": "hist"
}

n_components = Y_train.shape[1]

Y_pred_train = np.zeros_like(Y_train)
Y_pred_test  = np.zeros_like(Y_test)

# ==============================
# TRAIN PER PCA COMPONENT
# ==============================
print("\nTraining XGBoost (PCA space)...")

for i in tqdm(range(n_components)):

    model = XGBRegressor(**params)

    y_tr = Y_train[:, i]

    model.fit(X_train, y_tr)

    Y_pred_train[:, i] = model.predict(X_train)
    Y_pred_test[:, i]  = model.predict(X_test)

# ==============================
# METRICS
# ==============================
print("\n===== XGBOOST RESULTS (SUB-03) =====")

print("R2 Train:", r2_score(Y_train, Y_pred_train))
print("R2 Test :", r2_score(Y_test, Y_pred_test))

print("MSE Train:", mean_squared_error(Y_train, Y_pred_train))
print("MSE Test :", mean_squared_error(Y_test, Y_pred_test))

print("\nDONE ✅")

Shapes: (6002, 512) (6002, 200)

Training XGBoost (PCA space)...


100%|██████████| 200/200 [33:11<00:00,  9.96s/it]


===== XGBOOST RESULTS (SUB-03) =====
R2 Train: 0.08891738951206207
R2 Test : -0.013136991299688816
MSE Train: 188.4257049560547
MSE Test : 158.12554931640625

DONE ✅
